In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import glob
import json
import _utils

In [2]:
from statsmodels.distributions.empirical_distribution import ECDF

def ecdf_func(ensemble, obs):
    # np.nan is treated as Inf by ECDF, so need to manually remove these
    if np.isnan(ensemble).all():
        return(np.nan)
    if np.isnan(obs):
        return(np.nan)
    ensemble = ensemble[~np.isnan(ensemble)]
    
    return (ECDF(ensemble)(obs))

def ecdf_xr(ensemble, obs):       
    return xr.apply_ufunc(ecdf_func, ensemble, obs, 
                          input_core_dims = (["sim"], []), 
                          dask = "allowed", 
                          vectorize = True, ## required when function can only take 1D array
                         )

In [3]:
def test_ecdf(model_trends, obs_trend):
    ecdf_dat = []
    for mv in model_trends.sim.values:
        a = xr.concat([model_trends.drop(labels = mv, dim = "sim"), obs_trend.expand_dims({"sim": ["obs"]})], dim = "sim")
        b = model_trends.sel(sim = mv)
        x = ecdf_xr(a, b)
        ecdf_dat.append([(x == 0).sum().values, (x == 1).sum().values, (b > 0).sum().values, b.count().values])
    
    ecdf_dat = pd.DataFrame(ecdf_dat, columns = ["ecdf_0", "ecdf_1", "pos_trends", "num_values"])
    return(ecdf_dat)

In [4]:
dir = "../mnt_processed_data/"

In [6]:
model_var_dict = json.load(open(dir+"model_var_dict.json"))

In [6]:
## base time period, rx1day comparisons
start = 1979
end = 2020

for ensemble in ["cmip", "mesaclip", "spear"]:
    print(ensemble)
    model_trend = _utils.read_trends(dir, ensemble, "rx1day", start, end)
    
    for obs in ["cpc", "mswep"]:
        print(obs)
        obs_trend = _utils.read_trends(dir, obs, "rx1day", start, end)
        ecdf_result = ecdf_xr(model_trend, obs_trend)
        ecdf_result.to_netcdf(dir+"ecdf/"+obs+"_"+ensemble+"_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")

        if obs == "cpc":
            ecdf_test = test_ecdf(model_trend, obs_trend)
            ecdf_test.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"_rx1day_"+str(start)+"-"+str(end)+"_ecdf_test.csv")
            
        if ensemble == "cmip":
            ecdf_result = ecdf_xr(model_trend.sel(sim = model_var_dict["cmip_day_onevar"]), obs_trend)
            ecdf_result.to_netcdf(dir+"ecdf/"+obs+"_"+ensemble+"-sub_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")
            
            if obs == "cpc":
                ecdf_test = test_ecdf(model_trend.sel(sim = model_var_dict["cmip_day_onevar"]), obs_trend)
                ecdf_test.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"-sub_rx1day_"+str(start)+"-"+str(end)+"_ecdf_test.csv")


cmip
cpc
mswep
mesaclip
cpc
mswep
spear
cpc
mswep


In [7]:
## base time period, mon-p095 comparisons
start = 1979
end = 2020

for ensemble in ["cmip", "mesaclip", "spear"]:
    print(ensemble)
    model_trend = _utils.read_trends(dir, ensemble, "mon-p095", start, end)
    
    for obs in ["gpcc", "gpcp", "mswep"]:
        print(obs)
        obs_trend = _utils.read_trends(dir, obs, "mon-p095", start, end)
        ecdf_result = ecdf_xr(model_trend, obs_trend)
        ecdf_result.to_netcdf(dir+"ecdf/"+obs+"_"+ensemble+"_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

        if obs == "gpcc":
            ecdf_test = test_ecdf(model_trend, obs_trend)
            ecdf_test.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"_mon-p095_"+str(start)+"-"+str(end)+"_ecdf_test.csv")
                    
        if ensemble == "cmip":
            ecdf_result = ecdf_xr(model_trend.sel(sim = model_var_dict["cmip_mon_onevar"]), obs_trend)
            ecdf_result.to_netcdf(dir+"ecdf/"+obs+"_"+ensemble+"-sub_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

            if obs == "gpcc":
                ecdf_test = test_ecdf(model_trend, obs_trend)
                ecdf_test.to_csv(dir+"ecdf/"+obs+"_"+ensemble+"-sub_mon-p095_"+str(start)+"-"+str(end)+"_ecdf_test.csv")

cmip
gpcc
gpcp
mswep
mesaclip
gpcc
gpcp
mswep
spear
gpcc
gpcp
mswep


In [8]:
## loop through different time periods for mon-p095
for start in np.arange(1930, 1980, 5):  
    print(start)
    ## compare gpcc to models (cmip, mesaclip, spear)
    end = start+41
    gpcc_trend = _utils.read_trends(dir, "gpcc", "mon-p095", start, end)
    cmip_trend = _utils.read_trends(dir, "cmip", "mon-p095", start, end).sel(sim = model_var_dict["cmip_mon_onevar"])
    ecdf_result = ecdf_xr(cmip_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_cmip-sub_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

    mesaclip_trend = _utils.read_trends(dir, "mesaclip", "mon-p095", start, end)
    ecdf_result = ecdf_xr(mesaclip_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_mesaclip_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

    spear_trend = _utils.read_trends(dir, "spear", "mon-p095", start, end)
    ecdf_result = ecdf_xr(spear_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_spear_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

    ### with 2020 as end date
    end = 2020
    gpcc_trend = _utils.read_trends(dir, "gpcc", "mon-p095", start, end)
    cmip_trend = _utils.read_trends(dir, "cmip", "mon-p095", start, end).sel(sim = model_var_dict["cmip_mon_onevar"])
    ecdf_result = ecdf_xr(cmip_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_cmip-sub_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

    mesaclip_trend = _utils.read_trends(dir, "mesaclip", "mon-p095", start, end)
    ecdf_result = ecdf_xr(mesaclip_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_mesaclip_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

    spear_trend = _utils.read_trends(dir, "spear", "mon-p095", start, end)
    ecdf_result = ecdf_xr(spear_trend, gpcc_trend)
    ecdf_result.to_netcdf(dir+"ecdf/gpcc_spear_mon-p095_"+str(start)+"-"+str(end)+"_ecdf.nc")

In [30]:
## loop through different time periods for rx1day
for start in np.arange(1950, 1980, 5):  
    print(start)
    ## compare regen to models (cmip, mesaclip, spear)
    end = start+41
    regen_trend = _utils.read_trends(dir, "regen", "rx1day", start, end)
    cmip_trend = _utils.read_trends(dir, "cmip", "rx1day", start, end).sel(sim = model_var_dict["cmip_day_onevar"])
    ecdf_result = ecdf_xr(cmip_trend, regen_trend)
    ecdf_result.to_netcdf(dir+"ecdf/regen_cmip-sub_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")

    mesaclip_trend = _utils.read_trends(dir, "mesaclip", "rx1day", start, end)
    ecdf_result = ecdf_xr(mesaclip_trend, regen_trend)
    ecdf_result.to_netcdf(dir+"ecdf/regen_mesaclip_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")

    spear_trend = _utils.read_trends(dir, "spear", "rx1day", start, end)
    ecdf_result = ecdf_xr(spear_trend, regen_trend)
    ecdf_result.to_netcdf(dir+"ecdf/regen_spear_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")

    ### with 2016 as end date
    end = 2016
    regen_trend = _utils.read_trends(dir, "regen", "rx1day", start, end)
    cmip_trend = _utils.read_trends(dir, "cmip", "rx1day", start, end).sel(sim = model_var_dict["cmip_day_onevar"])
    ecdf_result = ecdf_xr(cmip_trend, regen_trend)
    ecdf_result.to_netcdf(dir+"ecdf/regen_cmip-sub_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")

    mesaclip_trend = _utils.read_trends(dir, "mesaclip", "rx1day", start, end)
    ecdf_result = ecdf_xr(mesaclip_trend, regen_trend)
    ecdf_result.to_netcdf(dir+"ecdf/regen_mesaclip_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")

    spear_trend = _utils.read_trends(dir, "spear", "rx1day", start, end)
    ecdf_result = ecdf_xr(spear_trend, regen_trend)
    ecdf_result.to_netcdf(dir+"ecdf/regen_spear_rx1day_"+str(start)+"-"+str(end)+"_ecdf.nc")
    

1950


KeyError: "not all values found in index 'sim'"

In [12]:
start

1950

In [13]:
end

1991

In [35]:
cmip_trend = _utils.read_trends(dir, "cmip", "rx1day", start, end)

In [40]:
for m in model_var_dict["cmip_day_onevar"]:
    cmip_trend.sel(sim = m)

KeyError: "not all values found in index 'sim'. Try setting the `method` keyword argument (example: method='nearest')."

In [41]:
m

'UKESM1-0-LL_r1i1p1f2'

In [16]:
cmip_trend.sim.values

array(['ACCESS-CM2_r10i1p1f1', 'ACCESS-CM2_r1i1p1f1',
       'ACCESS-CM2_r2i1p1f1', 'ACCESS-CM2_r3i1p1f1',
       'ACCESS-CM2_r4i1p1f1', 'ACCESS-CM2_r5i1p1f1',
       'ACCESS-CM2_r6i1p1f1', 'ACCESS-CM2_r7i1p1f1',
       'ACCESS-CM2_r8i1p1f1', 'ACCESS-CM2_r9i1p1f1',
       'ACCESS-ESM1-5_r10i1p1f1', 'ACCESS-ESM1-5_r11i1p1f1',
       'ACCESS-ESM1-5_r12i1p1f1', 'ACCESS-ESM1-5_r13i1p1f1',
       'ACCESS-ESM1-5_r14i1p1f1', 'ACCESS-ESM1-5_r15i1p1f1',
       'ACCESS-ESM1-5_r16i1p1f1', 'ACCESS-ESM1-5_r17i1p1f1',
       'ACCESS-ESM1-5_r18i1p1f1', 'ACCESS-ESM1-5_r19i1p1f1',
       'ACCESS-ESM1-5_r1i1p1f1', 'ACCESS-ESM1-5_r20i1p1f1',
       'ACCESS-ESM1-5_r21i1p1f1', 'ACCESS-ESM1-5_r22i1p1f1',
       'ACCESS-ESM1-5_r23i1p1f1', 'ACCESS-ESM1-5_r24i1p1f1',
       'ACCESS-ESM1-5_r25i1p1f1', 'ACCESS-ESM1-5_r26i1p1f1',
       'ACCESS-ESM1-5_r27i1p1f1', 'ACCESS-ESM1-5_r28i1p1f1',
       'ACCESS-ESM1-5_r29i1p1f1', 'ACCESS-ESM1-5_r2i1p1f1',
       'ACCESS-ESM1-5_r30i1p1f1', 'ACCESS-ESM1-5_r31i1p1f1',
    

In [17]:
xr.open_dataset("../mnt_processed_data/cmip_trends/

1950

In [31]:
ds = "cmip"

In [32]:
freq = "rx1day"

In [33]:
files = sorted(glob.glob(dir+ds+"_trends/"+ds+"_"+freq+"_*_"+str(start)+"-"+str(end)+"_trend.nc"))

In [34]:
files

['../mnt_processed_data/cmip_trends/cmip_rx1day_ACCESS-CM2_r10i1p1f1_1950-1991_trend.nc',
 '../mnt_processed_data/cmip_trends/cmip_rx1day_ACCESS-CM2_r1i1p1f1_1950-1991_trend.nc',
 '../mnt_processed_data/cmip_trends/cmip_rx1day_ACCESS-CM2_r2i1p1f1_1950-1991_trend.nc',
 '../mnt_processed_data/cmip_trends/cmip_rx1day_ACCESS-CM2_r3i1p1f1_1950-1991_trend.nc',
 '../mnt_processed_data/cmip_trends/cmip_rx1day_ACCESS-CM2_r4i1p1f1_1950-1991_trend.nc',
 '../mnt_processed_data/cmip_trends/cmip_rx1day_ACCESS-CM2_r5i1p1f1_1950-1991_trend.nc',
 '../mnt_processed_data/cmip_trends/cmip_rx1day_ACCESS-CM2_r6i1p1f1_1950-1991_trend.nc',
 '../mnt_processed_data/cmip_trends/cmip_rx1day_ACCESS-CM2_r7i1p1f1_1950-1991_trend.nc',
 '../mnt_processed_data/cmip_trends/cmip_rx1day_ACCESS-CM2_r8i1p1f1_1950-1991_trend.nc',
 '../mnt_processed_data/cmip_trends/cmip_rx1day_ACCESS-CM2_r9i1p1f1_1950-1991_trend.nc',
 '../mnt_processed_data/cmip_trends/cmip_rx1day_ACCESS-ESM1-5_r10i1p1f1_1950-1991_trend.nc',
 '../mnt_process

In [18]:
end

1991

In [ ]:
cmip_trend.sel(sim = model_var_dict["cmip_day_onevar"])